# 02_工具与结构化输出

**来源:**
- https://docs.langchain.com/docs/tools
- https://docs.langchain.com/docs/structured-output

## 1. 工具概述

**Tools（工具）** 扩展了 Agent 的能力——让它们可以获取实时数据、执行代码、查询外部数据库以及在现实世界中执行操作。

工具本质上是具有良好定义输入和输出的可调用函数，被传递给聊天模型。模型根据对话上下文决定何时调用工具以及提供什么参数。

## 2. 定义工具

### 2.1 使用 @tool 装饰器

最简单的方式是使用 `@tool` 装饰器：

In [ ]:
from langchain.tools import tool

# 最基本的工具定义
@tool
def search_database(query: str, limit: int = 10) -> str:
    """在客户数据库中搜索匹配查询的记录。

    参数:
        query: 搜索关键词
        limit: 返回的最大结果数
    """
    return f"找到 '{query}' 的 {limit} 条结果"

print(f"工具名称: {search_database.name}")
print(f"参数: {search_database.args}")
print(f"描述: {search_database.description}")

**命名规范：**
- 使用蛇形命名法（snake_case），例如 `web_search`
- 避免空格和特殊字符
- **类型提示是必需的**——它们定义了工具的输入模式
- 文档字符串应简洁明了，帮助模型理解工具的用途

### 2.2 自定义工具名称和描述

```python
# 自定义名称
@tool("web_search")
def search(query: str) -> str:
    """搜索网络信息"""
    return f"搜索 '{query}' 的结果"

# 自定义描述
@tool("calculator", description="执行算术计算。遇到数学问题时使用此工具。")
def calc(expression: str) -> str:
    """计算数学表达式"""
    return str(eval(expression))
```

### 2.3 高级模式定义

使用 Pydantic 模型或 JSON Schema 定义复杂输入：

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain.tools import tool

# 使用 Pydantic 定义输入模式
class WeatherInput(BaseModel):
    """天气查询的输入参数"""
    location: str = Field(description="城市名称或坐标")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="温度单位偏好"
    )
    include_forecast: bool = Field(
        default=False,
        description="是否包含 5 天预报"
    )

@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """获取当前天气和可选的预报"""
    temp = 22 if units == "celsius" else 72
    result = f"{location}当前天气: {temp}度{'摄' if units == 'celsius' else '华'}"
    if include_forecast:
        result += "\n未来5天: 晴朗"
    return result

print(f"工具模式: {get_weather.args_schema}")

## 3. 访问运行时上下文

工具通过 `ToolRuntime` 参数访问运行时信息，该参数对模型不可见（自动注入，隐藏在工具模式中）：

In [ ]:
from langchain.tools import tool, ToolRuntime

# 访问短期记忆（状态）
@tool
def get_last_user_message(runtime: ToolRuntime) -> str:
    """获取最近一条用户消息"""
    messages = runtime.state["messages"]
    for message in reversed(messages):
        if message.type == "human":
            return message.content
    return "未找到用户消息"

print(f"运行时工具（模型看不到 'runtime' 参数）: {get_weather.args}")

### ToolRuntime 组件

| 组件 | 描述 | 用途 |
|------|------|------|
| `state` | 短期记忆——当前对话的可变数据 | 访问对话历史、跟踪工具调用次数 |
| `context` | 不可变的运行时配置 | 用户 ID、会话信息、个性化 |
| `store` | 长期记忆——跨对话持久化 | 保存用户偏好、知识库 |
| `stream_writer` | 实时更新写入器 | 长操作进度反馈 |
| `execution_info` | 当前执行的身份和重试信息 | 线程/运行 ID、重试状态 |
| `tool_call_id` | 当前工具调用的唯一标识 | 日志关联和模型调用 |

### 从 State 读写

```python
from langchain.agents import AgentState
from langchain.messages import ToolMessage
from langchain.tools import ToolRuntime, tool
from langgraph.types import Command

class CustomState(AgentState):
    user_name: str

@tool
def set_user_name(new_name: str, runtime: ToolRuntime[None, CustomState]) -> Command:
    """在对话状态中设置用户名"""
    return Command(
        update={
            "user_name": new_name,
            "messages": [
                ToolMessage(
                    content=f"用户名已设置为 {new_name}。",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )
```

### 使用 Context（运行时配置）

```python
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime

@dataclass
class UserContext:
    user_id: str

@tool
def get_account_info(runtime: ToolRuntime[UserContext]) -> str:
    """获取当前用户的账户信息"""
    user_id = runtime.context.user_id
    # 查询数据库...
    return f"用户 {user_id} 的账户信息"
```

### 使用 Store（长期记忆）

```python
from langgraph.store.memory import InMemoryStore
from langchain.tools import tool, ToolRuntime

@tool
def get_user_info(user_id: str, runtime: ToolRuntime) -> str:
    """查询用户信息"""
    store = runtime.store
    user_info = store.get(("users",), user_id)
    return str(user_info.value) if user_info else "未知用户"

@tool
def save_user_info(user_id: str, user_info: dict, runtime: ToolRuntime) -> str:
    """保存用户信息"""
    store = runtime.store
    store.put(("users",), user_id, user_info)
    return "用户信息保存成功"
```

### Stream Writer（流式写入）

```python
@tool
def get_weather(city: str, runtime: ToolRuntime) -> str:
    """获取天气信息"""
    writer = runtime.stream_writer
    # 在工具执行过程中发送实时更新
    writer(f"正在查询城市: {city}")
    writer(f"获取到城市数据: {city}")
    return f"{city}的天气总是晴朗！"
```

## 4. 结构化输出

结构化输出允许 Agent 以特定、可预测的格式返回数据。直接在 Agent 的 `response_format` 参数中指定模式即可。

### 模式选择策略

| 策略 | 类 | 适用场景 |
|------|------|--------|
| **ProviderStrategy** | 提供商原生 | OpenAI、Anthropic、xAI (Grok) 等支持原生结构化输出的模型 |
| **ToolStrategy** | 工具调用 | 所有支持工具调用的模型（大多数现代模型） |
| 直接传入 Schema | 自动选择 | 传入 Pydantic/TypedDict 类型时自动选择最佳策略 |

### 4.1 ProviderStrategy（提供商原生）

某些模型提供商通过其 API 原生支持结构化输出，这是最可靠的方式：

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

# 定义输出模式
class ContactInfo(BaseModel):
    """联系信息提取"""
    name: str = Field(description="人员姓名")
    email: str = Field(description="电子邮件地址")
    phone: str = Field(description="电话号码")

# 创建 Agent 并指定输出格式（自动选择 ProviderStrategy）
agent = create_agent(
    model="openai:gpt-5.4",  # OpenAI 支持原生结构化输出
    response_format=ContactInfo,  # 直接传入模式类型
)

# 调用 Agent
result = agent.invoke({
    "messages": [{"role": "user", "content": "提取联系信息: 张三, zhangsan@example.com, (555) 123-4567"}]
})

print(f"结构化输出: {result['structured_response']}")
print(f"姓名: {result['structured_response'].name}")
print(f"邮箱: {result['structured_response'].email}")

### 4.2 ToolStrategy（工具调用）

对于不支持原生结构化输出的模型，LangChain 使用工具调用实现相同结果：

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

class ProductReview(BaseModel):
    """产品评论分析"""
    rating: int | None = Field(description="产品评分", ge=1, le=5)
    sentiment: Literal["positive", "negative"] = Field(description="评论情感")
    key_points: list[str] = Field(description="评论要点，小写，每点1-3个词")

# 使用 ToolStrategy
agent = create_agent(
    model="openai:gpt-5.4",
    response_format=ToolStrategy(ProductReview),
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "分析这条评论: '优秀产品: 5星满分。发货快，但价格贵'"}]
})

review = result["structured_response"]
print(f"结构化输出: {review}")
print(f"评分: {review.rating}")
print(f"情感: {review.sentiment}")
print(f"要点: {review.key_points}")

### 4.3 错误处理

`ToolStrategy` 的 `handle_errors` 参数控制验证失败时的重试策略：

```python
# 默认：捕获所有错误
ToolStrategy(schema=ProductReview)

# 自定义错误消息
ToolStrategy(
    schema=ProductReview,
    handle_errors="请提供有效的评分(1-5)和评论内容。"
)

# 仅处理特定异常
ToolStrategy(
    schema=ProductRating,
    handle_errors=ValueError  # 仅对 ValueError 重试
)

# 自定义错误处理函数
ToolStrategy(
    schema=Union[ContactInfo, EventDetails],
    handle_errors=lambda e: f"格式错误: {str(e)}"
)

# 禁用重试
ToolStrategy(schema=ProductRating, handle_errors=False)
```

### 4.4 联合类型（Union Types）

支持多个输出模式，模型根据上下文选择最合适的：

In [ ]:
from typing import Union
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

class ProductReview(BaseModel):
    """产品评论分析"""
    rating: int | None = Field(description="产品评分", ge=1, le=5)
    sentiment: Literal["positive", "negative"] = Field(description="评论情感")
    key_points: list[str] = Field(description="评论要点")

class CustomerComplaint(BaseModel):
    """客户投诉"""
    issue_type: Literal["product", "service", "shipping", "billing"] = Field(description="问题类型")
    severity: Literal["low", "medium", "high"] = Field(description="严重程度")
    description: str = Field(description="投诉简述")

# 联合类型：模型自动选择最匹配的模式
agent = create_agent(
    model="openai:gpt-5.4",
    response_format=ToolStrategy(Union[ProductReview, CustomerComplaint]),
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "分析这条评论: '优秀产品: 5星满分。发货快'"}]
})
print(f"自动选择的模式: {type(result['structured_response']).__name__}")
print(f"输出: {result['structured_response']}")

---

**参考链接**
- https://docs.langchain.com/docs/tools
- https://docs.langchain.com/docs/structured-output
- https://docs.langchain.com/docs/concepts/tool_calling
- https://docs.langchain.com/docs/how-tools-work